# 20. Cloud Data Warehouses, MPP & Apache Iceberg: Beginner Guide

### 📝 Universal SQL Execution Order (All SQL Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline (All 12 Clauses) ────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. QUALIFY         ➔ 8. SELECT & CASE      ➔ 9. DISTINCT (Dedup)           │
│ ➔ 10. UNION/INTERSECT➔ 11. ORDER BY (Sort)   ➔ 12. LIMIT / OFFSET (Page)     │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **20. Cloud Data Warehouses, MPP & Apache Iceberg**. Modern cloud analytical data platforms (Snowflake, Google BigQuery, Amazon Redshift, DuckDB) decouple compute from storage and utilize columnar physical layouts with Massively Parallel Processing (MPP). This notebook covers columnar storage layouts, immutable micro-partitions, automatic partition pruning via min-max metadata, clustering keys, compute cluster auto-scaling, modern Lakehouse table formats (Apache Iceberg, Delta Lake, Apache Parquet), and high-speed embedded vectorized analytics with DuckDB.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Columnar Storage Layouts vs Row-Oriented OLTP Storage
- [x] 🔹 Immutable Micro-Partitions & Metadata-Driven Partition Pruning
- [x] 🔹 Clustering Keys & Physical Data Sorting
- [x] 🔹 Modern Open Lakehouse Formats (Apache Iceberg, Delta Lake, Parquet)
- [x] 🔍 Scenario: Designing an MPP Analytical Query over Billions of Ledger Rows











In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Columnar Storage vs Row-Oriented Storage
- **What it does:** Columnar formats store each column sequentially on disk instead of packing entire rows together.
- **Key Note:** Slashing I/O by 95% because analytical queries reading only 3 columns out of 50 columns never touch the unreferenced columns on disk.
- **Dataset Application & Code Demonstration:** Compares row vs column physical storage concepts.


In [2]:
%%sql
SELECT 
    'Row-Oriented (Postgres/MySQL)' AS architecture, 'OLTP (Single-row CRUD)' AS optimal_workload, 'Reads entire row off disk' AS io_behavior, 'High write concurrency' AS benefit
UNION ALL
SELECT 'Columnar (Snowflake/BigQuery/DuckDB)', 'OLAP (Analytical Aggregations)', 'Reads only referenced columns', 'Huge compression (Snappy/ZSTD) + SIMD scan';


,architecture,optimal_workload,io_behavior,benefit
0,Row-Oriented (Postgres/MySQL),OLTP (Single-row CRUD),Reads entire row off disk,High write concurrency
1,Columnar (Snowflake/BigQuery/DuckDB),OLAP (Analytical Aggregations),Reads only referenced columns,Huge compression (Snappy/ZSTD) + SIMD scan


### 🔹 Micro-Partition Pruning & Metadata Min-Max Filtering
- **What it does:** Cloud warehouses automatically segment data into immutable micro-partitions (50MB–500MB) and store min/max statistics for every column in metadata.
- **Key Note:** Queries with `WHERE transaction_date BETWEEN ...` inspect metadata first and bypass downloading 99% of micro-partitions without indexes.
- **Dataset Application & Code Demonstration:** Simulates metadata-based partition pruning.


In [3]:
%%sql
SELECT 
    'Micro-Partition #001' AS partition_id, '2024-01-01' AS min_date, '2024-01-31' AS max_date, 500000 AS row_count, 'PRUNED (Skipped)' AS status_for_query_2024_03
UNION ALL
SELECT 'Micro-Partition #002', '2024-02-01', '2024-02-28', 500000, 'PRUNED (Skipped)'
UNION ALL
SELECT 'Micro-Partition #003', '2024-03-01', '2024-03-31', 500000, 'SCANNED';


,partition_id,min_date,max_date,row_count,status_for_query_2024_03
0,Micro-Partition #001,2024-01-01,2024-01-31,500000,PRUNED (Skipped)
1,Micro-Partition #002,2024-02-01,2024-02-28,500000,PRUNED (Skipped)
2,Micro-Partition #003,2024-03-01,2024-03-31,500000,SCANNED


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Open Table Formats: Apache Iceberg vs Delta Lake vs Apache Hudi
- **Objective:** Understand how modern open Lakehouse table formats provide ACID transactions and time-travel over cloud object storage (S3/GCS/Azure Blob).
- **Approach:** Construct an enterprise Lakehouse feature comparison matrix.


In [4]:
%%sql
SELECT 
    'Apache Iceberg' AS lakehouse_format, 'Snapshot-based Metadata Trees' AS metadata_engine, 'Full (SQL Standard)' AS acid_support, 'Multi-Engine Native (Snowflake/Spark/Trino/DuckDB)' AS ecosystem
UNION ALL
SELECT 'Delta Lake', 'JSON Transaction Log + Checkpoints', 'Full (ACID + Time Travel)', 'Databricks / Spark Centric'
UNION ALL
SELECT 'Apache Hudi', 'Timeline Service + Avro Payloads', 'Full (Streaming Ingestion)', 'Near-Real-Time Stream Processing';


,lakehouse_format,metadata_engine,acid_support,ecosystem
0,Apache Iceberg,Snapshot-based Metadata Trees,Full (SQL Standard),Multi-Engine Native (Snowflake/Spark/Trino/Duc...
1,Delta Lake,JSON Transaction Log + Checkpoints,Full (ACID + Time Travel),Databricks / Spark Centric
2,Apache Hudi,Timeline Service + Avro Payloads,Full (Streaming Ingestion),Near-Real-Time Stream Processing
